In this basic analysis I want to look at different ball park statistics for the 2025 season. Things such as batting average per stadium and batting average etc. Then I want to do similar analysis with  

In [1]:
import pandas as pd
import matplotlib.pyplot as plt  # data viz
import seaborn as sns            # data viz
import numpy as np
import pybaseball as pyb
from pybaseball import batting_stats, pitching_stats, statcast
import warnings
import pandas as pd
warnings.filterwarnings("ignore") 

#to view entire column
pd.set_option('display.max_colwidth', None)

In [2]:
from pybaseball import statcast

df = statcast(start_dt="2025-03-27", end_dt="2025-10-01")

This is a large query, it may take a moment to complete


100%|████████████████████████████████████████████████████████████████████████████████| 189/189 [01:44<00:00,  1.81it/s]


In [26]:
df.head()

,pitch_type,game_date,release_speed,release_pos_x,release_pos_z,player_name,batter,pitcher,events,description,...,batter_days_until_next_game,api_break_z_with_gravity,api_break_x_arm,api_break_x_batter_in,arm_angle,attack_angle,attack_direction,swing_path_tilt,intercept_ball_minus_batter_pos_x_inches,intercept_ball_minus_batter_pos_y_inches
605,CH,2024-10-01,88.1,-1.65,6.12,"Brieske, Beau",518792,689225,field_out,hit_into_play,...,1,2.5,1.4,-1.4,46.4,21.481595,-17.851272,24.228579,43.236189,31.25396
628,CH,2024-10-01,87.1,-1.69,6.17,"Brieske, Beau",518792,689225,NaN,swinging_strike,...,1,2.45,1.19,-1.19,44.1,14.52903,-35.809638,19.463062,61.68396,44.906197
641,CH,2024-10-01,89.7,-1.89,6.14,"Brieske, Beau",518792,689225,NaN,ball,...,1,2.44,1.33,-1.33,47.4,<NA>,<NA>,<NA>,<NA>,<NA>
666,FF,2024-10-01,97.5,-1.51,6.32,"Brieske, Beau",518792,689225,NaN,foul,...,1,0.82,0.4,-0.4,55.0,7.804441,0.738715,22.363442,50.109329,18.509113
698,CH,2024-10-01,88.6,-1.77,6.19,"Brieske, Beau",518792,689225,NaN,blocked_ball,...,1,2.32,1.3,-1.3,47.0,<NA>,<NA>,<NA>,<NA>,<NA>


In [9]:
list(df.columns)

['pitch_type',
 'game_date',
 'release_speed',
 'release_pos_x',
 'release_pos_z',
 'player_name',
 'batter',
 'pitcher',
 'events',
 'description',
 'spin_dir',
 'spin_rate_deprecated',
 'break_angle_deprecated',
 'break_length_deprecated',
 'zone',
 'des',
 'game_type',
 'stand',
 'p_throws',
 'home_team',
 'away_team',
 'type',
 'hit_location',
 'bb_type',
 'balls',
 'strikes',
 'game_year',
 'pfx_x',
 'pfx_z',
 'plate_x',
 'plate_z',
 'on_3b',
 'on_2b',
 'on_1b',
 'outs_when_up',
 'inning',
 'inning_topbot',
 'hc_x',
 'hc_y',
 'tfs_deprecated',
 'tfs_zulu_deprecated',
 'umpire',
 'sv_id',
 'vx0',
 'vy0',
 'vz0',
 'ax',
 'ay',
 'az',
 'sz_top',
 'sz_bot',
 'hit_distance_sc',
 'launch_speed',
 'launch_angle',
 'effective_speed',
 'release_spin_rate',
 'release_extension',
 'game_pk',
 'fielder_2',
 'fielder_3',
 'fielder_4',
 'fielder_5',
 'fielder_6',
 'fielder_7',
 'fielder_8',
 'fielder_9',
 'release_pos_y',
 'estimated_ba_using_speedangle',
 'estimated_woba_using_speedangle',
 'w

In [12]:
dfteams = df[['home_team', 'away_team', 'events']].dropna()

In [13]:
dfteams

,home_team,away_team,events
345,CLE,DET,field_out
393,CLE,DET,fielders_choice
470,CLE,DET,strikeout
614,CLE,DET,hit_by_pitch
781,CLE,DET,walk
...,...,...,...
2031,SD,ATL,force_out
2356,SD,ATL,walk
2885,SD,ATL,walk
3496,SD,ATL,field_out


In [17]:
df['events'].unique

In [21]:
# Only keep rows with actual plate appearances
pa_events = [
    'single', 'double', 'triple', 'home_run', 'walk', 'strikeout',
    'field_out', 'force_out', 'grounded_into_double_play',
    'field_error', 'hit_by_pitch', 'sac_fly', 'sac_bunt', 'other_out'
]

df_pa = df[df['events'].isin(pa_events)]

# Hits
df_pa['is_hit'] = df_pa['events'].isin(['single', 'double', 'triple', 'home_run'])

# At-bats = PA excluding walks, HBP, sac fly/bunt
df_pa['is_ab'] = ~df_pa['events'].isin(['walk', 'hit_by_pitch', 'sac_fly', 'sac_bunt'])

# Group by park
park_stats = df_pa.groupby('home_team').agg(
    hits=('is_hit', 'sum'),
    at_bats=('is_ab', 'sum'),
    home_runs=('is_hr', 'sum')
).reset_index()

park_stats['batting_avg'] = park_stats['hits'] / park_stats['at_bats']

In [26]:
# Highest batting average parks
park_stats.sort_values('batting_avg', ascending=False)

,home_team,hits,at_bats,home_runs,batting_avg
8,COL,1669,5692,217,0.293219
0,ATH,1468,5600,230,0.262143
29,WSH,1421,5492,177,0.258740
4,BOS,1410,5494,158,0.256644
20,PHI,1406,5485,217,0.256335
17,MIN,1401,5492,177,0.255098
2,AZ,1398,5490,183,0.254645
25,STL,1373,5412,143,0.253695
26,TB,1364,5386,207,0.253249
28,TOR,1367,5410,223,0.252680


In [25]:
# Most HR parks
park_stats.sort_values('home_runs', ascending=False)

,home_team,hits,at_bats,home_runs,batting_avg
14,LAD,1369,5522,259,0.247917
0,ATH,1468,5600,230,0.262143
3,BAL,1375,5453,228,0.252155
19,NYY,1258,5425,228,0.231889
13,LAA,1327,5428,226,0.244473
28,TOR,1367,5410,223,0.252680
8,COL,1669,5692,217,0.293219
20,PHI,1406,5485,217,0.256335
10,DET,1359,5404,208,0.251480
26,TB,1364,5386,207,0.253249


In [ ]:
# graphics

# PITCHING